# 05. Tool 整合

學習如何定義和整合工具到 LangGraph。

---

## 🎯 學習目標

完成本章節後，您將能夠：
- ✅ 定義不同類型的工具函數
- ✅ 建立工具調度器 (Tool Dispatcher)
- ✅ 實作多工具串聯執行
- ✅ 處理工具執行錯誤

---

## 📊 工具整合架構

```
┌─────────────────────────────────────────────────────────┐
│                    工具整合架構                          │
├─────────────────────────────────────────────────────────┤
│                                                         │
│   ┌─────────────────────────────────────────────────┐  │
│   │                 工具註冊表                        │  │
│   ├─────────────────────────────────────────────────┤  │
│   │  calculator  │  get_time  │  search  │  ...    │  │
│   └───────┬──────┴─────┬──────┴────┬─────┴─────────┘  │
│           │            │           │                   │
│           ▼            ▼           ▼                   │
│   ┌─────────────────────────────────────────────────┐  │
│   │              工具調度器 (Dispatcher)             │  │
│   │          根據名稱調用對應的工具函數               │  │
│   └─────────────────────────────────────────────────┘  │
│                                                         │
└─────────────────────────────────────────────────────────┘
```

In [1]:
from typing import TypedDict, Annotated, Literal
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from datetime import datetime

---

## 5.1 定義工具函數

### 工具函數規範

| 元素 | 說明 |
|------|------|
| 輸入 | 字串或簡單類型 |
| 輸出 | 字串（方便 LLM 理解） |
| Docstring | 描述工具用途（LLM 會讀取） |
| 錯誤處理 | 返回錯誤訊息而非拋出異常 |

In [2]:
def calculator(expression: str) -> str:
    """計算數學表達式
    
    範例輸入: "2 + 3 * 4"
    範例輸出: "計算結果: 14"
    """
    try:
        # 安全檢查：只允許數學運算
        allowed = set('0123456789+-*/(). ')
        if not all(c in allowed for c in expression):
            return "錯誤: 表達式包含不允許的字元"
        result = eval(expression)
        return f"計算結果: {result}"
    except Exception as e:
        return f"計算錯誤: {e}"

def get_time() -> str:
    """取得當前時間
    
    無需輸入參數
    """
    now = datetime.now()
    return f"當前時間: {now.strftime('%Y-%m-%d %H:%M:%S')}"

def search(query: str) -> str:
    """搜尋資訊（模擬）
    
    範例輸入: "Python 教學"
    """
    return f"搜尋 '{query}' 的結果: 找到 3 筆相關資料（模擬結果）"

def weather(city: str) -> str:
    """查詢天氣（模擬）
    
    範例輸入: "台北"
    """
    import random
    temp = random.randint(15, 35)
    conditions = ["晴天", "多雲", "陰天", "小雨"]
    return f"{city} 天氣: {random.choice(conditions)}，氣溫 {temp}°C"

# 工具註冊表
TOOLS = {
    "calculator": calculator,
    "get_time": get_time,
    "search": search,
    "weather": weather
}

print(f"📦 已註冊 {len(TOOLS)} 個工具:")
for name, fn in TOOLS.items():
    doc = fn.__doc__.split('\n')[0] if fn.__doc__ else "無描述"
    print(f"  • {name}: {doc}")

📦 已註冊 4 個工具:
  • calculator: 計算數學表達式
  • get_time: 取得當前時間
  • search: 搜尋資訊（模擬）
  • weather: 查詢天氣（模擬）


---

## 5.2 單一工具調度器

In [3]:
class ToolState(TypedDict):
    """單一工具執行狀態"""
    tool_name: str     # 要執行的工具名稱
    tool_input: str    # 工具輸入
    result: str        # 執行結果
    success: bool      # 是否成功

def execute_tool(state: ToolState) -> dict:
    """工具調度器：根據名稱執行對應工具"""
    tool_name = state["tool_name"]
    tool_input = state["tool_input"]
    
    print(f"  🔧 調用工具: {tool_name}")
    print(f"     輸入: {tool_input or '(無)'}")
    
    if tool_name not in TOOLS:
        return {"result": f"未知工具: {tool_name}", "success": False}
    
    try:
        tool_fn = TOOLS[tool_name]
        # 判斷工具是否需要參數
        if tool_input:
            result = tool_fn(tool_input)
        else:
            result = tool_fn()
        print(f"     結果: {result}")
        return {"result": result, "success": True}
    except Exception as e:
        return {"result": f"執行錯誤: {e}", "success": False}

graph = StateGraph(ToolState)
graph.add_node("execute", execute_tool)
graph.add_edge(START, "execute")
graph.add_edge("execute", END)

tool_app = graph.compile()
print("\n✅ 工具調度器已就緒")


✅ 工具調度器已就緒


In [4]:
# 測試各種工具
print("📊 測試工具調度器：")
print("=" * 50)

tests = [
    {"tool_name": "calculator", "tool_input": "(10 + 5) * 3"},
    {"tool_name": "get_time", "tool_input": ""},
    {"tool_name": "weather", "tool_input": "台北"},
    {"tool_name": "unknown", "tool_input": "test"},
]

for test in tests:
    result = tool_app.invoke({**test, "result": "", "success": False})
    status = "✅" if result["success"] else "❌"
    print(f"  {status} {result['result']}")
    print()

📊 測試工具調度器：
  🔧 調用工具: calculator
     輸入: (10 + 5) * 3
     結果: 計算結果: 45
  ✅ 計算結果: 45

  🔧 調用工具: get_time
     輸入: (無)
     結果: 當前時間: 2025-12-06 19:24:39
  ✅ 當前時間: 2025-12-06 19:24:39

  🔧 調用工具: weather
     輸入: 台北
     結果: 台北 天氣: 多雲，氣溫 31°C
  ✅ 台北 天氣: 多雲，氣溫 31°C

  🔧 調用工具: unknown
     輸入: test
  ❌ 未知工具: unknown



---

## 5.3 多工具串聯執行

處理多個工具的循環執行：

In [5]:
class MultiToolState(TypedDict):
    """多工具執行狀態"""
    pending_tools: list          # 待執行的工具列表
    results: Annotated[list, lambda a, b: a + b]  # 累加結果
    current_index: int           # 當前執行索引

def process_next_tool(state: MultiToolState) -> dict:
    """處理下一個工具"""
    idx = state["current_index"]
    tools = state["pending_tools"]
    
    if idx >= len(tools):
        return {"current_index": idx}
    
    tool_call = tools[idx]
    tool_name = tool_call["name"]
    tool_input = tool_call.get("input", "")
    
    print(f"  [{idx + 1}/{len(tools)}] 執行 {tool_name}")
    
    if tool_name in TOOLS:
        try:
            result = TOOLS[tool_name](tool_input) if tool_input else TOOLS[tool_name]()
        except Exception as e:
            result = f"錯誤: {e}"
    else:
        result = f"未知工具: {tool_name}"
    
    return {
        "results": [{"tool": tool_name, "result": result}],
        "current_index": idx + 1
    }

def should_continue(state: MultiToolState) -> Literal["continue", "end"]:
    """檢查是否還有工具要執行"""
    if state["current_index"] < len(state["pending_tools"]):
        return "continue"
    return "end"

multi_graph = StateGraph(MultiToolState)
multi_graph.add_node("process", process_next_tool)
multi_graph.add_edge(START, "process")
multi_graph.add_conditional_edges("process", should_continue, {
    "continue": "process",
    "end": END
})

multi_app = multi_graph.compile()
print("✅ 多工具執行器已就緒")

✅ 多工具執行器已就緒


In [6]:
print("📊 測試多工具串聯：")
print("=" * 50)

result = multi_app.invoke({
    "pending_tools": [
        {"name": "get_time"},
        {"name": "calculator", "input": "100 / 4"},
        {"name": "weather", "input": "台北"},
        {"name": "search", "input": "LangGraph 教學"}
    ],
    "results": [],
    "current_index": 0
})

print("\n" + "=" * 50)
print("📋 執行結果：")
for r in result["results"]:
    print(f"  • {r['tool']}: {r['result']}")

📊 測試多工具串聯：
  [1/4] 執行 get_time
  [2/4] 執行 calculator
  [3/4] 執行 weather
  [4/4] 執行 search

📋 執行結果：
  • get_time: 當前時間: 2025-12-06 19:24:39
  • calculator: 計算結果: 25.0
  • weather: 台北 天氣: 晴天，氣溫 30°C
  • search: 搜尋 'LangGraph 教學' 的結果: 找到 3 筆相關資料（模擬結果）


---

## 5.4 工具錯誤處理

In [7]:
def safe_execute(tool_fn, *args, **kwargs):
    """安全執行工具，捕捉所有錯誤
    
    Returns:
        dict: {"success": bool, "result": str, "error": str|None}
    """
    try:
        result = tool_fn(*args, **kwargs)
        return {"success": True, "result": result, "error": None}
    except Exception as e:
        return {"success": False, "result": None, "error": str(e)}

# 定義一個可能失敗的工具
def risky_tool(value: str) -> str:
    """可能失敗的工具"""
    num = int(value)
    if num <= 0:
        raise ValueError("值必須大於 0")
    return f"成功: {num * 2}"

# 測試安全執行
print("📊 測試安全執行：")
print("-" * 40)

tests = ["5", "-1", "abc"]
for t in tests:
    result = safe_execute(risky_tool, t)
    if result["success"]:
        print(f"  ✅ 輸入 '{t}' → {result['result']}")
    else:
        print(f"  ❌ 輸入 '{t}' → 錯誤: {result['error']}")

📊 測試安全執行：
----------------------------------------
  ✅ 輸入 '5' → 成功: 10
  ❌ 輸入 '-1' → 錯誤: 值必須大於 0
  ❌ 輸入 'abc' → 錯誤: invalid literal for int() with base 10: 'abc'


---

## 💡 重點回顧

### 工具設計原則

| 原則 | 說明 |
|------|------|
| 單一職責 | 每個工具只做一件事 |
| 清晰描述 | Docstring 描述用途和範例 |
| 字串輸出 | 方便 LLM 理解結果 |
| 錯誤處理 | 返回錯誤訊息而非拋異常 |

### 工具調度流程

```
1. 接收工具名稱和輸入
2. 從註冊表查找工具函數
3. 安全執行工具
4. 返回結果或錯誤
```

### 常見工具類型

- 🔢 **計算類**: 數學運算、單位轉換
- 🔍 **搜尋類**: 網路搜尋、資料庫查詢
- 🌐 **API 類**: 天氣、股票、翻譯
- 📁 **檔案類**: 讀寫檔案、格式轉換

---

## 📝 練習題

1. **新增工具**：實作一個貨幣轉換工具
2. **工具鏈**：讓一個工具的輸出成為另一個的輸入
3. **重試機制**：當工具失敗時自動重試 3 次
4. **工具限制**：限制每個工具每分鐘最多執行 10 次

---

下一步：[06. 條件路由](06_conditional_routing.ipynb)